# Delta Hedging in GBM and Merton Jump-Diffusion

This notebook studies **discrete-time delta hedging** for a European call option when the underlying follows:

- **Geometric Brownian Motion (GBM)** – the standard Black–Scholes–Merton (BSM) setting.
- **Merton Jump-Diffusion (MJD)** – GBM plus Poisson jumps in the log-price.

We:

- Implement price path generators for GBM and MJD (reusing the project stochastic engines).
- Derive and implement **BSM call price and delta** at a low level.
- Build a **delta-hedging agent** that rebalances a self-financing portfolio along each simulated path.
- Compare hedging performance (error distributions and risk metrics) between GBM and MJD.
- Visualise the results (histograms, density plots, example paths, error vs terminal price, etc.).


In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Match `01_market_simulation.ipynb` approach: add project root to path
sys.path.append('..')

from src.config_loader import load_config
from src.stochastic_engines import GBMModel, MertonJumpDiffusion
from src.analytics import bsm_price, bsm_delta

plt.style.use('seaborn-v0_8')
sns.set_context('talk')

config = load_config()  # reads `config/parameters.yaml`

# Unify config keys with what `stochastic_engines.py` expects
config.market = config.market_params
config.merton = config.merton_params
config.sim = config.simulation

S0 = config.market.S0
r = config.market.r
sigma = config.market.sigma
T = config.market.T
dt = config.market.dt
n_paths = config.sim.n_paths
seed = config.sim.seed

# Merton jump parameters (used for Merton-delta hedging)
lambda_j = config.merton.lmbda
mu_j = config.merton.mu_j
sigma_j = config.merton.sigma_j
# kappa = E[exp(J)] - 1 where J is the log-jump
kappa = np.exp(mu_j + 0.5 * sigma_j**2) - 1

# Truncate Poisson series for Merton closed-form pricing/delta
n_merton_terms = 20

# Option contract parameters
K = S0  # at-the-money call
option_type = 'call'

n_steps = int(T / dt)
T_grid = np.linspace(0.0, T, n_steps + 1)

S0, r, sigma, T, dt, n_paths

In [ ]:
gbm_engine = GBMModel(config)
mjd_engine = MertonJumpDiffusion(config)

gbm_paths = gbm_engine.generate_paths()
mjd_paths = mjd_engine.generate_paths()

gbm_paths.shape, mjd_paths.shape

## Delta-Hedging Agent (BSM)

We implement a **self-financing delta-hedging strategy**:

- At time 0, we price the option with BSM and set the initial portfolio value to this price.
- We choose the initial delta from BSM and solve for the required cash position.
- At each rebalancing time step:
  - The stock position and cash position evolve over \(dt\).
  - We recompute the BSM delta using the current spot and remaining time to maturity.
  - We trade stock and adjust cash so the portfolio remains self-financing.
- At maturity, we compare the terminal portfolio value to the option payoff to obtain the **hedging error** per path.


In [ ]:
def payoff_call(S_T, K):
    return np.maximum(S_T - K, 0.0)


def simulate_bsm_delta_hedge(paths, K, r, sigma_hedge, T, dt, option_type='call'):
    """Simulate *BSM* (GBM) delta hedging on given price paths."""

    n_paths, n_steps_plus = paths.shape
    n_steps = n_steps_plus - 1

    # Time-to-maturity grid (remaining time at each node)
    t_grid = np.linspace(0.0, T, n_steps_plus)
    tau_grid = T - t_grid

    S0 = paths[:, 0]

    # Initial option price and delta (BSM)
    C0 = bsm_price(S0, K, T, r, sigma_hedge, option_type=option_type)
    delta0 = bsm_delta(S0, K, T, r, sigma_hedge, option_type=option_type)

    # Portfolio state variables
    V = np.zeros_like(paths)
    delta = np.zeros_like(paths)
    cash = np.zeros_like(paths)

    V[:, 0] = C0
    delta[:, 0] = delta0
    cash[:, 0] = V[:, 0] - delta[:, 0] * S0

    disc = np.exp(r * dt)

    for k in range(n_steps):
        S_tp1 = paths[:, k + 1]

        # Portfolio before re-hedging at t_{k+1}
        V_tp1 = delta[:, k] * S_tp1 + cash[:, k] * disc

        # Remaining time to maturity at t_{k+1}
        tau_tp1 = tau_grid[k + 1]

        # New delta from BSM
        new_delta = bsm_delta(S_tp1, K, tau_tp1, r, sigma_hedge, option_type=option_type)

        # Self-financing: adjust cash after changing stock position
        cash_tp1 = V_tp1 - new_delta * S_tp1

        V[:, k + 1] = V_tp1
        delta[:, k + 1] = new_delta
        cash[:, k + 1] = cash_tp1

    S_T = paths[:, -1]
    if option_type != 'call':
        raise NotImplementedError("Only call options are implemented in this notebook.")

    payoff = payoff_call(S_T, K)

    terminal_portfolio = V[:, -1]
    hedging_error = terminal_portfolio - payoff

    return hedging_error, terminal_portfolio, payoff


def _poisson_pmf_weights(mean: float, n_max: int) -> np.ndarray:
    """Poisson PMF weights via stable recursion."""
    weights = np.zeros(n_max + 1, dtype=float)
    weights[0] = np.exp(-mean)
    for n in range(1, n_max + 1):
        weights[n] = weights[n - 1] * mean / n
    return weights


def merton_call_price(
    S,
    K,
    T,
    r,
    sigma,
    lambda_j,
    mu_j,
    sigma_j,
    n_max=20,
    option_type='call',
):
    """Merton jump-diffusion European call price using a Poisson-mixture of BSM."""
    if option_type != 'call':
        raise NotImplementedError("Only call pricing is implemented in this notebook.")

    S_arr = np.asarray(S, dtype=float)
    out = np.zeros_like(S_arr, dtype=float)

    if T <= 0:
        return payoff_call(S_arr, K)

    kappa = np.exp(mu_j + 0.5 * sigma_j**2) - 1
    mean = lambda_j * T
    weights = _poisson_pmf_weights(mean, n_max)

    sigma2 = sigma**2
    for n, w in enumerate(weights):
        if w == 0.0:
            continue
        sigma_n = np.sqrt(sigma2 + n * (sigma_j**2) / T)
        r_n = r - lambda_j * kappa + n * (mu_j + 0.5 * sigma_j**2) / T
        out += w * bsm_price(S_arr, K, T, r_n, sigma_n, option_type='call')

    return out


def merton_delta(
    S,
    K,
    T,
    r,
    sigma,
    lambda_j,
    mu_j,
    sigma_j,
    n_max=20,
    option_type='call',
):
    """Merton delta derived as the Poisson-mixture of BSM deltas."""
    if option_type != 'call':
        raise NotImplementedError("Only call delta is implemented in this notebook.")

    S_arr = np.asarray(S, dtype=float)

    if T <= 0:
        return np.zeros_like(S_arr, dtype=float)

    kappa = np.exp(mu_j + 0.5 * sigma_j**2) - 1
    mean = lambda_j * T
    weights = _poisson_pmf_weights(mean, n_max)

    sigma2 = sigma**2
    out = np.zeros_like(S_arr, dtype=float)
    for n, w in enumerate(weights):
        if w == 0.0:
            continue
        sigma_n = np.sqrt(sigma2 + n * (sigma_j**2) / T)
        r_n = r - lambda_j * kappa + n * (mu_j + 0.5 * sigma_j**2) / T
        out += w * bsm_delta(S_arr, K, T, r_n, sigma_n, option_type='call')

    return out


def simulate_merton_delta_hedge(
    paths,
    K,
    r,
    sigma,
    lambda_j,
    mu_j,
    sigma_j,
    T,
    dt,
    n_max=20,
    option_type='call',
):
    """Simulate *Merton* (jump) delta hedging on given price paths."""
    n_paths, n_steps_plus = paths.shape
    n_steps = n_steps_plus - 1

    # Time-to-maturity grid (remaining time at each node)
    t_grid = np.linspace(0.0, T, n_steps_plus)
    tau_grid = T - t_grid

    S0 = paths[:, 0]

    if option_type != 'call':
        raise NotImplementedError("Only call options are implemented in this notebook.")

    V0 = merton_call_price(S0, K, T, r, sigma, lambda_j, mu_j, sigma_j, n_max=n_max, option_type='call')
    delta0 = merton_delta(S0, K, T, r, sigma, lambda_j, mu_j, sigma_j, n_max=n_max, option_type='call')

    # Portfolio state variables
    V = np.zeros_like(paths)
    delta = np.zeros_like(paths)
    cash = np.zeros_like(paths)

    V[:, 0] = V0
    delta[:, 0] = delta0
    cash[:, 0] = V[:, 0] - delta[:, 0] * S0

    disc = np.exp(r * dt)

    for k in range(n_steps):
        S_tp1 = paths[:, k + 1]

        # Portfolio before re-hedging at t_{k+1}
        V_tp1 = delta[:, k] * S_tp1 + cash[:, k] * disc

        # Remaining time to maturity at t_{k+1}
        tau_tp1 = tau_grid[k + 1]

        # New delta from Merton
        new_delta = merton_delta(
            S_tp1,
            K,
            tau_tp1,
            r,
            sigma,
            lambda_j,
            mu_j,
            sigma_j,
            n_max=n_max,
            option_type='call',
        )

        # Self-financing: adjust cash after changing stock position
        cash_tp1 = V_tp1 - new_delta * S_tp1

        V[:, k + 1] = V_tp1
        delta[:, k + 1] = new_delta
        cash[:, k + 1] = cash_tp1

    S_T = paths[:, -1]
    payoff = payoff_call(S_T, K)

    terminal_portfolio = V[:, -1]
    hedging_error = terminal_portfolio - payoff

    return hedging_error, terminal_portfolio, payoff


In [ ]:
# Run delta-hedging experiments for 2 hedging agents x 2 underlying models

hedge_sigma = sigma  # GBM/BSM hedger uses the diffusive volatility

# Underlying = GBM
gbm_bsm_error, gbm_bsm_V_T, gbm_payoff_1 = simulate_bsm_delta_hedge(
    gbm_paths, K, r, hedge_sigma, T, dt, option_type=option_type
)

gbm_merton_error, gbm_merton_V_T, gbm_payoff_2 = simulate_merton_delta_hedge(
    gbm_paths,
    K,
    r,
    sigma,
    lambda_j,
    mu_j,
    sigma_j,
    T,
    dt,
    n_max=n_merton_terms,
    option_type=option_type,
)

# Underlying = Merton jump-diffusion
mjd_bsm_error, mjd_bsm_V_T, mjd_payoff_1 = simulate_bsm_delta_hedge(
    mjd_paths, K, r, hedge_sigma, T, dt, option_type=option_type
)

mjd_merton_error, mjd_merton_V_T, mjd_payoff_2 = simulate_merton_delta_hedge(
    mjd_paths,
    K,
    r,
    sigma,
    lambda_j,
    mu_j,
    sigma_j,
    T,
    dt,
    n_max=n_merton_terms,
    option_type=option_type,
)

# Sanity check: payoff should match for the same underlying (hedging agent only changes V_T)
print('GBM payoff match:', np.allclose(gbm_payoff_1, gbm_payoff_2))
print('Merton payoff match:', np.allclose(mjd_payoff_1, mjd_payoff_2))

(
    gbm_bsm_error.mean(),
    gbm_merton_error.mean(),
    mjd_bsm_error.mean(),
    mjd_merton_error.mean(),
)

In [ ]:
# Collect results into a DataFrame for easier analysis and plotting

def summarise_errors(df: pd.DataFrame) -> pd.Series:
    err = df["hedging_error"]
    abs_err = df["abs_hedging_error"]
    rel_err = df["rel_hedging_error"]

    # CVaR (expected tail loss) on absolute error
    q95 = abs_err.quantile(0.95)
    cvar_abs_95 = abs_err[abs_err >= q95].mean() if len(abs_err) else np.nan

    return pd.Series(
        {
            "mean": err.mean(),
            "std": err.std(ddof=1),
            "rmse": np.sqrt(np.mean(err**2)),
            "mae": abs_err.mean(),
            "median": err.quantile(0.50),
            "q05": err.quantile(0.05),
            "q95": err.quantile(0.95),
            "prob_err_gt0": (err > 0).mean(),
            "cvar_abs_95": cvar_abs_95,
            "rel_rmse": np.sqrt(np.mean(rel_err**2)),
        }
    )

results = pd.DataFrame(
    {
        "underlying": np.concatenate(
            [
                np.repeat("GBM", n_paths),
                np.repeat("GBM", n_paths),
                np.repeat("Merton", n_paths),
                np.repeat("Merton", n_paths),
            ]
        ),
        "hedge_agent": np.concatenate(
            [
                np.repeat("BSM", n_paths),
                np.repeat("Merton", n_paths),
                np.repeat("BSM", n_paths),
                np.repeat("Merton", n_paths),
            ]
        ),
        "S_T": np.concatenate(
            [
                gbm_paths[:, -1],
                gbm_paths[:, -1],
                mjd_paths[:, -1],
                mjd_paths[:, -1],
            ]
        ),
        "payoff": np.concatenate(
            [
                gbm_payoff_1,
                gbm_payoff_1,
                mjd_payoff_1,
                mjd_payoff_1,
            ]
        ),
        "V_T": np.concatenate(
            [
                gbm_bsm_V_T,
                gbm_merton_V_T,
                mjd_bsm_V_T,
                mjd_merton_V_T,
            ]
        ),
        "hedging_error": np.concatenate(
            [
                gbm_bsm_error,
                gbm_merton_error,
                mjd_bsm_error,
                mjd_merton_error,
            ]
        ),
    }
)

results["abs_hedging_error"] = np.abs(results["hedging_error"])

# Relative error: stabilise division for near-zero payoffs
# (call payoff can be 0 for many paths)
results["rel_hedging_error"] = results["hedging_error"] / results["payoff"].clip(lower=1e-8)

results["group"] = results["underlying"] + " | " + results["hedge_agent"]

summary = results.groupby(["underlying", "hedge_agent"]).apply(summarise_errors)
summary

In [ ]:
# Visualisations of hedging performance

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Distribution of hedging error
sns.kdeplot(
    data=results,
    x="hedging_error",
    hue="group",
    fill=True,
    ax=axes[0, 0],
    common_norm=False,
)
axes[0, 0].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[0, 0].set_title("Hedging Error Distribution")
axes[0, 0].set_xlabel("V_T - Payoff")

# 2. Boxplot of hedging errors
sns.boxplot(data=results, x="group", y="hedging_error", ax=axes[0, 1])
axes[0, 1].set_title("Hedging Error Boxplot")
axes[0, 1].tick_params(axis='x', rotation=25)

# 3. Error vs terminal price
sns.scatterplot(
    data=results.sample(min(8000, len(results)), random_state=0),
    x="S_T",
    y="hedging_error",
    hue="group",
    alpha=0.35,
    ax=axes[1, 0],
)
axes[1, 0].axhline(0.0, color="black", linestyle="--", linewidth=1)
axes[1, 0].set_title("Hedging Error vs Terminal Price")

# 4. Terminal portfolio vs payoff (replication quality)
plot_df = results.sample(min(8000, len(results)), random_state=1)
max_val = float(np.max(np.concatenate([plot_df["payoff"].values, plot_df["V_T"].values])))

sns.scatterplot(
    data=plot_df,
    x="payoff",
    y="V_T",
    hue="group",
    alpha=0.35,
    ax=axes[1, 1],
)
axes[1, 1].plot([0.0, max_val], [0.0, max_val], linestyle='--', color='black', linewidth=1)
axes[1, 1].set_title("Terminal Portfolio vs Payoff")
axes[1, 1].set_xlabel("Payoff")
axes[1, 1].set_ylabel("V_T")

plt.tight_layout()
plt.show()
